In [30]:
from tnwater import load_gps, load_water_quality, merge_water_quality_with_gps, filter_ids


nut_df = load_water_quality("../data/wq_data_for_tennessee.csv")
gps = load_gps("../data/dam_distances.csv")
merged_df = merge_water_quality_with_gps(nut_df=nut_df, gps_df=gps)

# note that it converts any depths in feet to meters during the import



C:\Users\Spencer Womble\OneDrive\TN_Tech2\projects\reservoir_work\analysis_files\project_folder\tnwater\pipeline.py:29: DtypeWarning: Columns (7,15,16,18,22,24,26,28,29,30,31,32,47,48,49,50,51,52,53,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [2]:
# get ride of duplicate observations by random sampling (phosphorus seems to have a lot of duplicates)
group_keys = ['date_time', 'site', 'characteristic', 'sample_fraction']

deduped = (
    merged_df.sample(frac=1, random_state=42)              # shuffle all rows - important to do first as we shuffle and then keep the first row for each duplicate
      .drop_duplicates(subset=group_keys, keep='first')
      .sort_values('date_time')
      .reset_index(drop=True)
)

In [ ]:
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# !!! Fix the default values for 'ids' argument so they include all ids by default 
# (i.e., transfer the list of ids to the list in the preprocessing script) !!!

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


# filter down to the location IDs that have good data coverage. The default contains the location IDs, but you can manually specify them if you want to.gps

# I'm retaining all here
filtered_df = filter_ids(df=deduped, ids=['pp16', 'cent_h13',      'pp8',  'cent_h1',  'cent_h7',
      'pp6',     'pp11', 'cent_h16',     'dh17',     'dh16',      'dh8',
      'dh6', 'cent_h14',  'cent_h9', 'cent_h12',  'cent_h6',  'cent_h2',
  'cent_h8',  'cent_h5',  'cent_h3',     'pp15',     'pp12',      'pp4',
      'pp3',      'pp2',      'pp1',      'pp9',     'pp10',     'pp13',
     'pp14',      'pp7',     'dh10',     'dh15',      'dh4',      'dh2',
      'dh9',     'dh11',      'dh3',     'dh14',     'dh13',     'dh18',
     'dh12',      'dh1',      'dh5', 'cent_h20',  'cent_h4', 'cent_h11',
      'pp5', 'cent_h19', 'cent_h17', 'cent_h18', 'cent_h15', 'cent_h10',
      'dh7'])

In [4]:
# check if all nutrient/tss values are in mg/L
mask = (filtered_df["measure_unit_code"] == "mg/l") & (filtered_df["sample_fraction"].notna())
filtered_df[mask]["measure_unit_code"].unique()



array(['mg/l'], dtype=object)

In [ ]:
# check if any values are less than the detection limit and change them to half the detection limit if they are

# Boolean check: are there ever any rows where measure_value < quantitation_limit_value?
any_less = (filtered_df['measure_value'] < filtered_df['quantitation_limit_value']).any()
print(f"measure_value is ever less than quantitation_limit_value: {any_less}")

# there are a handful of observations below the detection limit, but only a couple for parameters we're actually using

measure_value is ever less than quantitation_limit_value: True


In [ ]:

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# !!!! Move function below into preprocessing script !!!!

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


# Replace values below the detection limit with half the detection limit
filtered_df['measure_value'] = filtered_df.apply(
    lambda row: row['quantitation_limit_value'] / 2
    if row['measure_value'] < row['quantitation_limit_value']
    else row['measure_value'],
    axis=1
)


# Filter to rows where measure_vaule < quantitation_limit_value to check that it worked
filtered_df[filtered_df['measure_value'] < filtered_df['quantitation_limit_value']].head()




,start_date,start_time,characteristic,measure_value,measure_unit_code,sample_fraction,quantitation_limit_value,analytical_method_identifier,result_speciation,latitude,...,month,decimal_date,site,agency,position,id,numeric_id,distance_from_dam_centerline_km,distance_from_centerline_km,total_distance
191325,2010-07-29,10:55:00,Organic nitrogen,0.250,mg/l,Total,0.50,351.2,as N,36.1553,...,7,2010.573849,percy_priest,tdec,reservoir,pp16,16.0,0.452,0.0,0.452
191329,2010-07-29,10:55:00,Kjeldahl nitrogen,0.250,mg/l,Total,0.50,351.2,as N,36.1553,...,7,2010.573849,percy_priest,tdec,reservoir,pp16,16.0,0.452,0.0,0.452
191335,2010-07-29,10:55:00,Phosphorus,0.025,mg/l,Total,0.05,4500-P-E,as P,36.1553,...,7,2010.573849,percy_priest,tdec,reservoir,pp16,16.0,0.452,0.0,0.452
243521,2012-07-26,11:45:00,Kjeldahl nitrogen,0.250,mg/l,Total,0.50,351.2,as N,36.1553,...,7,2012.566911,percy_priest,tdec,reservoir,pp16,16.0,0.452,0.0,0.452
243522,2012-07-26,11:45:00,Nitrite,0.025,mg/l,Total,0.05,300,as NO2,36.1553,...,7,2012.566911,percy_priest,tdec,reservoir,pp16,16.0,0.452,0.0,0.452


In [28]:
merged_df['activity_depth_value'].isna().sum()

np.int64(689368)

In [ ]:
# replace blanks/NAs for depth with 0 to indiciate that they are surface samples
# It's important to do this step before filtering to keep only the shallowest measurements for the depth profile samples
# as pandas will shift all NAs to the end, throwing the sorting order out of alignment

merged_df['activity_depth_value'] = merged_df['activity_depth_value'].fillna(0)

In [ ]:
# Task 1: Keep all single-sample rows + only the shallowest row from depth profiles
filtered_surface_df = (
    merged_df
    .sort_values(['id', 'date_time', 'activity_depth_value'])
    .drop_duplicates(subset=['id', 'date_time'], keep='first')
)



In [ ]:
# Task 2: Extract only rows from id/date_time combinations that have multiple depth measurements
depth_profiles_df = merged_df[
    merged_df
    .groupby(['id', 'date_time'])['activity_depth_value']
    .transform('size') > 1].copy()


depth_profiles_df.to_csv('../data/depth_profile_subset.csv', index = False)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Spencer to do 5-6-26: Need to address the depth-cutoff problem. We've now retained only the shallowest depth samples for the samples that were taken along a descending depth gradient. However, some of the retained samples were still collected quite deep (max = 56 m, most of the deeper samples collected between 4-7 m). We need to either set a removal threshold by defining what counts as a surface sample and removing samples collected at a deeper depth. Or, we need to leave as is and use depth as a covariate in the model.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [16]:
from tnwater import pivot_wider

# pivot data to wide format for plotting and modeling
filtered_df_wide = pivot_wider(filtered_df)


In [6]:
from tnwater import clean_column_names
# clean column names for the new nutrient variable columns

filtered_df_wide = clean_column_names(filtered_df_wide)



In [7]:
# check for outliers in no3_no2 column

from tnwater import outlier_check

outlier_check(filtered_df_wide, 
              sort_col='inorganic_nitrogen_(nitrate_and_nitrite)_total',
              keep_col='inorganic_nitrogen_(nitrate_and_nitrite)_total',
              rows = 5)

,inorganic_nitrogen_(nitrate_and_nitrite)_total
3760,549.000
2499,4.300
1998,4.000
1773,3.278
2121,3.200


In [8]:
# check for outliers in tp column

from tnwater import outlier_check

outlier_check(filtered_df_wide, 
              sort_col='phosphorus_total',
              keep_col='phosphorus_total',
              rows = 5)

,phosphorus_total
928,6.400
534,2.820
2570,1.300
3498,0.923
3792,0.908


In [9]:

# check for outliers in chlorophyll_a_total column

from tnwater import outlier_check


outlier_check(filtered_df_wide, 
              sort_col='chlorophyll_a_total',
              keep_col='chlorophyll_a_total',
              rows = 5)

,chlorophyll_a_total
2849,69.7
2776,50.7
3066,45.0
3244,43.8
3214,39.6


In [10]:
# check for outliers in tss column

from tnwater import outlier_check


outlier_check(filtered_df_wide, 
              sort_col='total_suspended_solids_suspended',
              keep_col='total_suspended_solids_suspended',
              rows = 5)

,total_suspended_solids_suspended
1812,79.0
1241,61.6
1578,55.7
2824,54.0
1278,45.5


In [ ]:
from tnwater import outlier_to_na

# outlier thresholds based on professional judgement and data exploration
# there were no obvious outliers for chla or tss - at least on the high end

# convert the outliers to NA for no3_no2
filtered_df=outlier_to_na(filtered_df_wide, 
                          col='inorganic_nitrogen_(nitrate_and_nitrite)_total',
                          greater_than_threshold=100)

# convert the outliers to NA for TP
filtered_df=outlier_to_na(filtered_df_wide, 
                          col='phosphorus_total',
                          greater_than_threshold=3)

,inorganic_nitrogen_(nitrate_and_nitrite)_total
2499,4.300
1998,4.000
1773,3.278
2121,3.200
2779,3.180


In [16]:
filtered_df.columns

Index(['date_time', 'site', 'id', 'position', 'year', 'month', 'decimal_date',
       'latitude', 'longitude', 'distance_from_centerline_km',
       'total_distance', 'activity_depth_value', 'ammonia_total',
       'ammonium_total', 'chlorophyll_a_total', 'chlorophyll_b_fixed',
       'chlorophyll_c_fixed', 'inorganic_nitrogen_(nitrate_and_nitrite)_total',
       'kjeldahl_nitrogen_total', 'nitrogen_total', 'organic_nitrogen_total',
       'orthophosphate_dissolved', 'phosphorus_dissolved', 'phosphorus_total',
       'total_suspended_solids_suspended', 'turbidity_total', 'ph_total'],
      dtype='object')

In [7]:
# write filtered wide data frame to new csv

filtered_df_wide.to_csv('../data/cleaned_data.csv', index=False)